In [0]:
-- 1. Read, Type, and Trim (Streaming, No DISTINCT)
CREATE TEMPORARY STREAMING LIVE VIEW stg_returns_typed AS
SELECT
  -- Handle date formatting anomalies and bad string values
  COALESCE(
    try_to_date(TRIM(return_date), 'M/d/yyyy'),
    try_to_date(TRIM(return_date), 'yyyy-M-d'),
    try_to_date(TRIM(return_date), 'd-MMM-yy'),
    CAST(TRIM(return_date) AS DATE)
  ) AS return_date,
  
  -- Casting bad string data (like "UNKNOWN") to INT automatically converts them to NULL
  CAST(TRIM(product_id) AS INT) AS product_id,
  CAST(TRIM(store_id) AS INT) AS store_id,
  CAST(TRIM(quantity) AS INT) AS quantity,
  
  ingested_timestamp,
  batch_date
FROM STREAM(mavenmarket.bronze.returns);

-- 2. Run Generic KPI Checks (Completeness, Validity, and Stream-Static RI)
CREATE TEMPORARY STREAMING LIVE VIEW stg_returns_checked AS
SELECT 
  r.*,
  
  -- HARD CHECKS (Route to Quarantine)
  CASE WHEN r.quantity IS NULL THEN 'Completeness: quantity is null; ' ELSE '' END AS err_qty_null,
  CASE WHEN r.quantity <= 0 THEN 'Validity: quantity must be greater than 0; ' ELSE '' END AS err_qty_neg,
  CASE WHEN r.store_id IS NULL THEN 'Completeness: store_id is null; ' ELSE '' END AS err_store_null,
  CASE WHEN r.product_id IS NULL THEN 'Completeness: product_id is null; ' ELSE '' END AS err_product_null,
  
  -- Referential Integrity: Stream-Static Join against dimension tables
  CASE WHEN r.product_id IS NOT NULL AND p.product_id IS NULL 
       THEN 'Referential Integrity: product_id not found in products; ' ELSE '' END AS err_fk_product,
  CASE WHEN r.store_id IS NOT NULL AND s.store_id IS NULL 
       THEN 'Referential Integrity: store_id not found in stores; ' ELSE '' END AS err_fk_store,
       
  -- WARN CHECKS (Stay in Silver, added to dq_flags)
  CASE WHEN r.return_date IS NULL THEN 'Completeness: return_date is null; ' ELSE '' END AS warn_date
FROM STREAM(live.stg_returns_typed) r
-- Left joins to validate the foreign keys without breaking the stream
LEFT JOIN mavenmarket.bronze.products p 
  ON r.product_id = CAST(p.product_id AS INT)
LEFT JOIN mavenmarket.bronze.stores s 
  ON r.store_id = CAST(s.store_id AS INT);

-- 3. Consolidate Flags
CREATE TEMPORARY STREAMING LIVE VIEW stg_returns_flagged AS
SELECT 
  *,
  TRIM(CONCAT(err_qty_null, err_qty_neg, err_store_null, err_product_null, err_fk_product, err_fk_store)) AS hard_failures,
  TRIM(CONCAT(warn_date)) AS dq_flags
FROM STREAM(live.stg_returns_checked);

-- 4. Write Target Tables (Split the stream)
-- CLEAN / WARN TABLE
CREATE STREAMING LIVE TABLE silver_returns
COMMENT "Cleaned returns fact table with warn-level data quality flags."
AS SELECT 
  * EXCEPT(err_qty_null, err_qty_neg, err_store_null, err_product_null, err_fk_product, err_fk_store, warn_date, hard_failures)
FROM STREAM(live.stg_returns_flagged)
WHERE hard_failures = '';

-- QUARANTINE TABLE
CREATE STREAMING LIVE TABLE quarantine_returns
COMMENT "Quarantined returns failing hard checks"
AS SELECT 
  * EXCEPT(err_qty_null, err_qty_neg, err_store_null, err_product_null, err_fk_product, err_fk_store, warn_date, dq_flags),
  hard_failures AS quarantine_reasons
FROM STREAM(live.stg_returns_flagged)
WHERE hard_failures != '';